# Conformal Prediction

Split conformal intervals with finite-sample coverage guarantees.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

from prob_ml.conformal.metrics import coverage, interval_width
from prob_ml.conformal.split import split_conformal_regression
from prob_ml.data.regression_dgp import RegressionDGPConfig, generate_regression_data
from prob_ml.models.trainer import TrainConfig

In [ ]:
data = generate_regression_data(RegressionDGPConfig(n_samples=3000, seed=42))
X_train, X_tmp, y_train, y_tmp = train_test_split(data.X, data.y, test_size=0.4, random_state=42)
X_cal, X_test, y_cal, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

result = split_conformal_regression(
    X_train, y_train, X_cal, y_cal, X_test,
    alpha=0.1,
    config=TrainConfig(epochs=40, seed=42),
)
cov = coverage(y_test, result.lower, result.upper)
width = interval_width(result.lower, result.upper)
print(f'Coverage={cov:.3f} (target 0.90), mean width={width:.3f}')

In [ ]:
idx = np.argsort(result.y_hat)[:80]
plt.errorbar(
    result.y_hat[idx], y_test[idx],
    yerr=[result.y_hat[idx] - result.lower[idx], result.upper[idx] - result.y_hat[idx]],
    fmt='o', alpha=0.6, capsize=2,
)
plt.xlabel('Prediction')
plt.ylabel('True y')
plt.title('Conformal intervals (subset)')
plt.show()